# ISOM 835 · Explainer — Naive Bayes, Plain and Simple
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**After this notebook you will be able to** compute a Naive Bayes churn prediction by hand with simple counts, check it against scikit-learn, and say when the method is a good choice and when its percentages should not be trusted.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. No formulas, no Greek letters. If you can count and multiply, you can do Naive Bayes.

### Words you will hear today

| Word | Plain English |
|---|---|
| **Base rate** | How common something is before we know anything about the customer. "6 of our 14 customers left" → base rate 6/14. |
| **Conditional share** ("given") | A share *inside one group*. "Of the 6 who left, 5 were month-to-month" → 5/6. Written P(month-to-month \| churn). |
| **Score** | Base rate × the conditional shares, one score per outcome (churn, stay). Bigger score wins. |
| **Normalize** | Turn the two scores into percentages that add up to 100%: divide each score by their total. |
| **Naive** | The method pretends the customer's features are unrelated to each other. That is the shortcut that makes it fast. |
| **Smoothing** | Pretend we saw one extra customer of each kind, so no share is ever exactly zero. |

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## 1. A counting game

You manage customer retention at a small internet provider. Here are **14 past customers**. For each one you know two things, plus whether they left:

- **Contract:** month-to-month, one year, or two year
- **TechSupport:** did they pay for tech support? (Yes / No)
- **Churn:** did they cancel? (Yes / No)

**Today's question:** a new customer just signed up. She is **month-to-month** with **no tech support**. Will she churn?

**What the next cell does:** types the 14 customers into a small table (a pandas DataFrame) and shows it. **How to read it:** one row per customer. Look at the Churn column and see if you can spot a pattern before we count anything.

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    'Contract': ['Month-to-month', 'One year', 'Month-to-month', 'Two year', 'Month-to-month', 'One year', 'Month-to-month',
                 'Two year', 'Month-to-month', 'One year', 'Month-to-month', 'Two year', 'One year', 'Month-to-month'],
    'TechSupport': ['No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'No',
                    'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'No'],
    'Churn': ['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes',
              'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes'],
})
customers.index = [f'C{i:02d}' for i in range(1, 15)]
customers

### The obvious idea: find customers exactly like her

The simplest answer is: look up past customers with the **same** contract and the **same** tech support, and see how many left.

**What the next cell does:** keeps only the month-to-month customers without tech support and counts them.

In [ ]:
same = customers[(customers['Contract'] == 'Month-to-month') & (customers['TechSupport'] == 'No')]
left = (same['Churn'] == 'Yes').sum()
print(f'customers exactly like her: {len(same)}')
print(f'of those, how many left:    {left}')

Four of the five customers exactly like her left. That works here because we only have **two** features.

**The problem:** a real company records 10, 20, 50 things about each customer. Ask for "month-to-month, no tech support, fiber internet, pays by electronic check, senior, no partner, ..." and you will usually find **zero** past customers who match on everything. You can't count an empty group.

**Naive Bayes gets around this.** Instead of finding customers who match on *everything at once*, it looks at **one feature at a time** and then multiplies the pieces together. Let's do it by hand.

## 2. Count it by hand

We need three kinds of counts.

**(a) The base rate.** Before we know anything about her, how common is churn?

**What the next cell does:** counts who left and who stayed. **How to read it:** the two counts, and the share of each.

In [ ]:
n_total = len(customers)
n_churn = (customers['Churn'] == 'Yes').sum()
n_stay = (customers['Churn'] == 'No').sum()

print(f'{n_churn} of the {n_total} customers left   → P(churn) = {n_churn}/{n_total} = {n_churn / n_total:.3f}')
print(f'{n_stay} of the {n_total} customers stayed → P(stay)  = {n_stay}/{n_total} = {n_stay / n_total:.3f}')

So **6 of 14 left** and **8 of 14 stayed**. If we knew nothing else, we would lean toward "stay."

**(b) Contract, inside each group.** Now split the customers into the two groups (left, stayed) and ask: within each group, how many were month-to-month?

**What the next cell does:** a `crosstab` counts customers by Contract (rows) and Churn (columns). **How to read it:** read **down a column**. The "Yes" column is the 6 customers who left; the "No" column is the 8 who stayed.

In [ ]:
contract_counts = pd.crosstab(customers['Contract'], customers['Churn'])
print(contract_counts)
print()

m2m_churn = contract_counts.loc['Month-to-month', 'Yes']
m2m_stay = contract_counts.loc['Month-to-month', 'No']
print(f'Of the {n_churn} customers who left,   {m2m_churn} were month-to-month → P(month-to-month | churn) = {m2m_churn}/{n_churn} = {m2m_churn / n_churn:.3f}')
print(f'Of the {n_stay} customers who stayed, {m2m_stay} were month-to-month → P(month-to-month | stay)  = {m2m_stay}/{n_stay} = {m2m_stay / n_stay:.3f}')

**In words:** of the 6 customers who left, **5** were month-to-month. Of the 8 who stayed, only **2** were. Month-to-month is much more common among people who leave, so it is evidence *for* churn.

**(c) Tech support, inside each group.** Same idea, second feature.

In [ ]:
support_counts = pd.crosstab(customers['TechSupport'], customers['Churn'])
print(support_counts)
print()

nots_churn = support_counts.loc['No', 'Yes']
nots_stay = support_counts.loc['No', 'No']
print(f'Of the {n_churn} customers who left,   {nots_churn} had no tech support → P(no support | churn) = {nots_churn}/{n_churn} = {nots_churn / n_churn:.3f}')
print(f'Of the {n_stay} customers who stayed, {nots_stay} had no tech support → P(no support | stay)  = {nots_stay}/{n_stay} = {nots_stay / n_stay:.3f}')

**In words:** of the 6 who left, **5** had no tech support. Of the 8 who stayed, **3** had none. More evidence for churn.

## 3. Multiply, compare, normalize

Now we build **one score per outcome**:

> **Churn score** = (share who left) × (share of leavers who were month-to-month) × (share of leavers with no support)
>
> **Stay score** = (share who stayed) × (share of stayers who were month-to-month) × (share of stayers with no support)

That's it. That is Naive Bayes. (The idea comes from **Bayes' rule**, which in words says: *how likely an outcome is after seeing the evidence depends on how common the outcome is to begin with, times how typical the evidence is for that outcome.*)

The two scores do not add up to 1, so in the last step we **normalize**: divide each score by the total of both.

**What the next cell does:** prints the arithmetic step by step. **How to read it:** compare the two scores, then read the final percentage.

In [ ]:
churn_score = (n_churn / n_total) * (m2m_churn / n_churn) * (nots_churn / n_churn)
stay_score = (n_stay / n_total) * (m2m_stay / n_stay) * (nots_stay / n_stay)

print('Step 1: multiply the pieces for each outcome')
print(f'  churn score = {n_churn}/{n_total} × {m2m_churn}/{n_churn} × {nots_churn}/{n_churn} = {churn_score:.4f}')
print(f'  stay score  = {n_stay}/{n_total} × {m2m_stay}/{n_stay} × {nots_stay}/{n_stay} = {stay_score:.4f}')
print()
print('Step 2: the bigger score wins')
print(f'  {churn_score:.4f} vs. {stay_score:.4f}  →  predict', 'CHURN' if churn_score > stay_score else 'STAY')
print()
print('Step 3: normalize so the two add up to 100%')
total = churn_score + stay_score
print(f'  P(churn) = {churn_score:.4f} / ({churn_score:.4f} + {stay_score:.4f}) = {churn_score / total:.1%}')
print(f'  P(stay)  = {stay_score:.4f} / ({churn_score:.4f} + {stay_score:.4f}) = {stay_score / total:.1%}')

**Result:** churn score **0.2976**, stay score **0.0536**. Naive Bayes says she has an **84.7%** chance of churning.

Notice what happened. The base rate leaned toward "stay" (8 of 14). But both features pointed strongly toward "churn," and multiplying the evidence overturned the base rate.

Compare with the direct lookup in Step 1: 4 of 5 identical customers left, which is 80%. Naive Bayes gave 84.7%. Close, but a bit **more extreme**. Remember that; it comes back in Step 8.

## 4. Why is it called "naive"?

When we multiplied the contract share by the tech-support share, we quietly assumed the two features are **unrelated**: that knowing someone is month-to-month tells you nothing about whether they bought tech support.

**In real life that is often wrong.** Month-to-month customers are usually the ones testing the service, and people testing a service rarely pay for add-ons like tech support. So "month-to-month" and "no tech support" tend to show up **together**. They are partly the **same signal** told twice.

Naive Bayes doesn't know that, so it counts the evidence **twice**. A business picture: two references for a job candidate who both work at the same company and talked to each other first. You get two glowing letters, but really one opinion. Treat them as independent and you become too sure of the candidate.

**Why it often still works:** double-counting pushes the percentages too far (84.7% instead of 80%), but it usually pushes them in the **right direction**. The customer most likely to leave still gets a higher score than the customer least likely to leave. So Naive Bayes is often good at **ranking** customers ("call these 100 first") even when its **percentages** are off. We will see both facts on real data.

## 5. Check our work with scikit-learn

scikit-learn's version for category features is called `CategoricalNB`. Two small setup steps:

- **`OrdinalEncoder`** turns text categories into numbers (for example Month-to-month → 0, One year → 1, Two year → 2), because scikit-learn models only take numbers. The numbers are just labels; their order means nothing here.
- **`alpha=1e-9`** turns smoothing off (almost exactly zero), so the model counts exactly like we did. More on alpha in a moment.

**What the next cell does:** trains `CategoricalNB` on the 14 customers and asks it about our new customer. **How to read it:** the model's churn probability should match our hand calculation.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

features = ['Contract', 'TechSupport']
encoder = OrdinalEncoder()
X_small = encoder.fit_transform(customers[features])
y_small = (customers['Churn'] == 'Yes').astype(int)

nb_exact = CategoricalNB(alpha=1e-9)
nb_exact.fit(X_small, y_small)

new_customer = pd.DataFrame({'Contract': ['Month-to-month'], 'TechSupport': ['No']})
x_new = encoder.transform(new_customer)

print(f'hand calculation:        P(churn) = {churn_score / total:.1%}')
print(f'scikit-learn (alpha≈0):  P(churn) = {nb_exact.predict_proba(x_new)[0, 1]:.1%}')

Same answer: **84.7%**. There is no magic inside `CategoricalNB`; it is our counting game.

### The zero problem, and why we smooth

Try a different new customer: **two-year contract, no tech support.** Look back at the table: of the 6 customers who left, **none** had a two-year contract. So the share is 0/6 = 0, and anything times zero is zero. The churn score is exactly **0**, and the model says **0% chance** of churning.

That is far too sure. We have only 14 customers; "we haven't seen it yet" is not the same as "it can never happen."

**Smoothing** fixes this: **pretend we saw one extra customer of each kind** in each group. One extra two-year leaver, one extra one-year leaver, one extra month-to-month leaver, and so on. Now no share is ever zero. In scikit-learn this is `alpha=1`, which is the default.

**What the next cell does:** scores the two-year customer with smoothing off and on, and shows the smoothed hand calculation. **How to read it:** with smoothing off you get exactly 0%; with it on you get a sensible, uncertain answer.

In [ ]:
two_year = pd.DataFrame({'Contract': ['Two year'], 'TechSupport': ['No']})
x_two = encoder.transform(two_year)

nb_smooth = CategoricalNB(alpha=1)
nb_smooth.fit(X_small, y_small)

print('Two-year contract, no tech support:')
print(f'  smoothing off (alpha≈0): P(churn) = {nb_exact.predict_proba(x_two)[0, 1]:.1%}')
print(f'  smoothing on  (alpha=1): P(churn) = {nb_smooth.predict_proba(x_two)[0, 1]:.1%}')
print()

# The smoothed hand calculation. Contract has 3 kinds, so each group gets 3 pretend customers for Contract.
# TechSupport has 2 kinds, so each group gets 2 pretend customers for TechSupport.
two_churn = contract_counts.loc['Two year', 'Yes']
two_stay = contract_counts.loc['Two year', 'No']
s_churn = (n_churn / n_total) * ((two_churn + 1) / (n_churn + 3)) * ((nots_churn + 1) / (n_churn + 2))
s_stay = (n_stay / n_total) * ((two_stay + 1) / (n_stay + 3)) * ((nots_stay + 1) / (n_stay + 2))
print('By hand, with one pretend customer of each kind:')
print(f'  churn score = {n_churn}/{n_total} × ({two_churn}+1)/({n_churn}+3) × ({nots_churn}+1)/({n_churn}+2) = {s_churn:.4f}')
print(f'  stay score  = {n_stay}/{n_total} × ({two_stay}+1)/({n_stay}+3) × ({nots_stay}+1)/({n_stay}+2) = {s_stay:.4f}')
print(f'  P(churn) = {s_churn / (s_churn + s_stay):.1%}')

With smoothing off the model says **0.0%**; with smoothing on it says **30.1%**, and our hand calculation agrees. "Probably stays, but we're not certain" is a much more honest answer from 14 customers.

**Rule of thumb:** keep the default `alpha=1`. We only turned it off above to show that the model really is counting.

## 6. Scale up: the real Telco customers

Same method, real data: about 7,000 customers from the Telco dataset. We use five category columns:

- **Contract**, **InternetService**, **PaymentMethod**, **TechSupport**, **OnlineSecurity**

**What the next cell does:** loads the data and splits it: 80% to **train** the model, 20% held back to **test** it, like an exam the model has not seen. `stratify` keeps the churn rate the same in both parts, and `random_state=835` makes the split the same every time you run it. **How to read it:** customer counts and the churn rate.

In [ ]:
from sklearn.model_selection import train_test_split

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL)

columns = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']
X = df[columns]
y = (df['Churn'] == 'Yes').astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)

print(f'{len(df):,} customers · {y.mean():.1%} churned')
print(f'{len(X_tr):,} to train · {len(X_te):,} to test · {y_te.sum()} churners in the test set')

### Quick check of the "naive" assumption on real data

In Step 4 we guessed that month-to-month customers rarely buy tech support. **What the next cell does:** for each contract type, shows the share of customers *without* tech support. **How to read it:** if the features were unrelated, the three numbers would be about the same.

In [ ]:
no_support_share = (df['TechSupport'] == 'No').groupby(df['Contract']).mean()
for contract, share in no_support_share.items():
    print(f'{contract:15s} {share:.0%} have no tech support')

**69%** of month-to-month customers have no tech support, against **14%** of two-year customers. The features are clearly related, so Naive Bayes will double-count some evidence. (There is a second overlap too: TechSupport and OnlineSecurity both contain a "No internet service" category, which repeats what InternetService already says.) Let's see whether it still works.

### Train Naive Bayes and compare with a "nobody churns" guess

A **baseline** is the dumbest reasonable guess. Here: predict that **nobody** churns. Since most customers stay, that guess is right most of the time, so any model has to beat it.

**What the next cell does:** encodes the categories, trains `CategoricalNB` with the default smoothing, and scores it on the test set. **How to read it:** accuracy of the model vs. the baseline, then the four boxes of the confusion matrix in words.

In [ ]:
from sklearn.metrics import confusion_matrix

encoder_big = OrdinalEncoder()
X_tr_enc = encoder_big.fit_transform(X_tr)
X_te_enc = encoder_big.transform(X_te)

nb = CategoricalNB()          # alpha=1, the default smoothing
nb.fit(X_tr_enc, y_tr)
pred = nb.predict(X_te_enc)

accuracy = (pred == y_te).mean()
baseline = (y_te == 0).mean()
print(f'Naive Bayes accuracy:         {accuracy:.1%}')
print(f'"Nobody churns" accuracy:     {baseline:.1%}')
print()

tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(f'Of the {tp + fn} test customers who really left:')
print(f'  the model flagged {tp} (caught) and missed {fn}.')
print(f'Of the {tn + fp} test customers who really stayed:')
print(f'  the model correctly left {tn} alone and wrongly flagged {fp}.')

**How to read this:** accuracy barely beats the baseline (**74.7%** vs. **73.5%**). That looks disappointing, but accuracy hides the useful part. The "nobody churns" guess catches **zero** churners. Naive Bayes caught **283 of the 374** customers who really left, at the price of flagging **266** customers who stayed. For a retention team, that list is the whole point. (Session 5 showed how to pick the cutoff with dollars instead of accuracy.)

## 7. How does it compare with logistic regression?

To compare models fairly we use **ROC-AUC** (from Session 4). In plain words:

> **AUC = if you pick one churner and one non-churner at random, how often does the model give the churner the higher score?**
>
> 0.5 = a coin flip. 1.0 = perfect every time.

AUC only cares about **ranking**, not about whether the percentages are right.

**What the next cell does:** trains logistic regression on the same five columns (with `OneHotEncoder`, which gives each category its own 0/1 column, the right input for logistic regression) and prints both AUCs.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

logit = make_pipeline(OneHotEncoder(), LogisticRegression(max_iter=2000))
logit.fit(X_tr, y_tr)

p_nb = nb.predict_proba(X_te_enc)[:, 1]
p_logit = logit.predict_proba(X_te)[:, 1]

print(f'Naive Bayes          AUC {roc_auc_score(y_te, p_nb):.3f}')
print(f'Logistic regression  AUC {roc_auc_score(y_te, p_logit):.3f}')

**0.810** vs. **0.819**. Naive Bayes ranks customers almost as well as logistic regression, even though its "unrelated features" assumption is wrong. That is the "often still ranks well" point from Step 4, on real data.

## 8. When to use Naive Bayes, and the one big caution

**Good uses:**
- **Fast.** Training is just counting, so it handles millions of rows in seconds.
- **Tiny data.** With a few dozen examples it often does as well as fancier models, because it has so few things to learn.
- **Text and spam filtering.** The classic use: each word is a feature ("free," "winner," "invoice"). With thousands of words, almost no two emails match exactly, so counting one word at a time is exactly the right trick.
- **A quick baseline.** Train it first. If a fancier model can't beat it, the fancier model isn't earning its keep.

**The big caution: the percentages are often too extreme (overconfident).** Double-counted evidence pushes scores toward 0% and 100%. **What the next cell does:** groups test customers by what Naive Bayes *said* and compares with what *really happened*. **How to read it:** if the model were honest, the two percentages in each row would be close.

In [ ]:
groups = pd.cut(p_nb, bins=[0, 0.1, 0.5, 0.9, 1.0], labels=['under 10%', '10% to 50%', '50% to 90%', 'over 90%'])
check = pd.DataFrame({'model said': p_nb, 'really churned': y_te.values, 'group': groups})
summary = check.groupby('group', observed=True).agg(customers=('model said', 'size'), model_said=('model said', 'mean'), really_churned=('really churned', 'mean'))
for name, row in summary.iterrows():
    print(f'NB said {name:11s}: {row.customers:4.0f} customers · model said {row.model_said:5.1%} on average · really churned {row.really_churned:5.1%}')
print()
print(f'Highest churn probability from logistic regression: {p_logit.max():.1%}')

Look at the last group: for **220** customers Naive Bayes said **93.3%** on average, but only **62.7%** really churned. At the low end it said **1.2%** for a group where **6.2%** churned. The ranking is fine; the **numbers** are stretched. Logistic regression never said anything above **64.5%**.

**Why it matters:** in Session 5 we multiplied churn probabilities by dollars (expected gain = probability × $450). Do that with raw Naive Bayes numbers and you will overestimate the value of contacting your "sure" churners by about 50%. **Before you multiply a Naive Bayes probability by dollars, check its calibration** (Session 5, Step 4) and fix it with `CalibratedClassifierCV` if needed. Using it only to **rank** customers is fine.

## 9. Your turn

1. **By hand, a different customer.** Using the 14-customer table, a new customer has a **one-year contract** and **does have tech support**. Without smoothing, compute her churn score, stay score, and P(churn).
   - *Hint:* you need four counts: one-year leavers, one-year stayers, leavers with support, stayers with support. Read them off `contract_counts` and `support_counts`. Then check your answer with `nb_exact.predict_proba(encoder.transform(pd.DataFrame({'Contract': ['One year'], 'TechSupport': ['Yes']})))`.
2. **Add a column.** Add `'PaperlessBilling'` to the `columns` list in Step 6 and re-run Steps 6–7. Did Naive Bayes's AUC go up or down? Did logistic regression's?
   - *Hint:* change one line, then Runtime → Run all.
3. **Turn the smoothing up.** On the real data, train `CategoricalNB(alpha=100)` instead of the default. Does the AUC change much? Why might smoothing matter less with 5,600 training customers than with 14?
   - *Hint:* how much does one pretend customer change a count of 1,000? How much does it change a count of 5?

In [ ]:
# Your turn — work here

## What we learned

- **Naive Bayes is a counting game:** base rate × the share of each feature inside each outcome group, then normalize the scores so they add to 100%.
- **It looks at one feature at a time,** so it works even when no past customer matches the new one on everything.
- **"Naive" means it treats features as unrelated.** That double-counts related evidence, which stretches the percentages, but the **ranking** is often still good (AUC 0.810 vs. 0.819 for logistic regression here).
- **Smoothing (alpha = 1) pretends we saw one extra customer of each kind,** so an unseen combination never gets an impossible 0%.
- **Use it when you need speed, have tiny data or text, or want a quick baseline,** and check calibration before you multiply its probabilities by dollars.

### Words to know

| Word | Plain English |
|---|---|
| **Base rate** | How common an outcome is before you know anything else. 6 of 14 left → 6/14. |
| **Conditional share** | A share inside one group. Of the 6 who left, 5 were month-to-month → 5/6. |
| **Score** | Base rate × conditional shares, one per outcome. Bigger wins. |
| **Normalize** | Divide each score by the total so they add to 100%. |
| **Naive (independence) assumption** | Pretending features are unrelated to each other. |
| **Smoothing (alpha)** | Pretend one extra customer of each kind so no share is zero. `alpha=1` is the default. |
| **`CategoricalNB`** | scikit-learn's Naive Bayes for category features. |
| **`OrdinalEncoder`** | Turns text categories into number labels a model can read. |
| **Baseline** | The simplest guess a model must beat, here "nobody churns." |
| **ROC-AUC** | Pick one churner and one non-churner at random: how often does the model rank the churner higher? |
| **Overconfident** | Percentages pushed too close to 0% or 100%. Fine for ranking, risky for dollar math. |